> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# IS 4487 Lab 14: API Integration

## Outline
1. Import customer reviews
2. Create prompts for LLM
3. Summarize Customer Reviews

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Labs/lab_14_api.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Customer Reviews - Business Context

This dataset represents customer reviews of Megatelco phones, which provides valuable insight into customer satisfaction, product performance, and brand perception. In a business context, companies use this type of data to understand how customers feel about their products, identify common issues (like battery life or software bugs), and detect what features customers value most (such as camera quality or speed). Because reviews are unstructured text, they contain rich, detailed feedback that goes beyond simple ratings—making them especially useful for improving products, guiding marketing strategies, and reducing customer churn.

In this lab, instead of using traditional text analysis methods, you will use a large language model (LLM) to analyze the reviews. Traditional text analytics techniques (like keyword counting or basic sentiment tools such as VADER) rely on predefined rules or dictionaries and often struggle with nuance, sarcasm, or context. LLMs, on the other hand, are trained on massive amounts of text and can understand language more like a human—recognizing tone, context, and subtle meaning. This allows them to perform more advanced tasks, such as summarizing feedback, detecting themes, interpreting sarcasm, and generating insights directly from raw text. Using an LLM enables businesses to extract deeper, more accurate insights from customer reviews at scale, making it a powerful tool for modern data analysis.

## Data Dictionary

| Column                        | Data Type       | Description                                                  |
|------------------------------|------------------|--------------------------------------------------------------|
| `Date`                   | Date           | Date of the review                                              |
| `Stars`                 | Integer           | Number of stars, from 1 (low) to 5 (high)                                     |
| `Review`             | String       | Text of the customer review                      |


## APIs for AI

An API for a large language model (LLM) allows applications to send text (like a question or prompt) to a powerful AI model hosted on a server and receive a generated response in return. Instead of building and running their own AI models, businesses can simply make requests to the API—often with just a few lines of code—and get capabilities like text generation, summarization, classification, or customer support automation.

This is useful for businesses because it makes advanced AI accessible without requiring deep technical expertise or expensive infrastructure. Companies can quickly integrate LLMs into their products and workflows to improve efficiency, enhance customer experiences (like chatbots or personalized responses), and gain insights from large amounts of text data, all while saving time and development costs.




## Part 1: Load the Data

### What you are going to do:
- Load the dataset
- Preview the data

**Things to notice:**
- Do you see any elements in the reviews that would difficult for VADER or other lexicon-based models to process?


In [25]:
import pandas as pd
import google.generativeai as genai

Create a dataframe containing sample reviews for use in the lab

In [26]:
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/megatelco_customer_reviews.csv"
df = pd.read_csv(url)

df.head()

,Date,Stars,Review
0,2025-12-01,4,"I purchased a Megatelco phone last week, and i..."
1,2025-12-03,3,"My Megatelco phone arrived fast, but I've noti..."
2,2025-12-05,5,Megatelco offers top-notch phones! I'm happy w...
3,2025-12-08,4,I've been using Megatelco phones for a while n...
4,2025-12-10,2,"Regrettably, my experience with Megatelco was ..."


# Part 2 : Prepare the LLM Prompt
### What you are going to do:
- Instruct the LLM on the context and desired output

### Why this matters:
A LLM needs to know what you are trying to accomplish, what data you will provide, what it should do with the data, and how to format the output.  Your prompt needs to set all of this context before passing in a review.  

In [27]:
base_prompt = (
    "Summarize the sentiment and most important points in the following user review for "
    "a phone company named Megatelco.  You will evaluate the sentiment, theme (in 2 words or less), count the number of words, and suggest an action (in 2 words or less)."
     "Format the output in a table with columns: "
     "Sentiment, Theme, Word Count, Suggested Action. Review: "
)

#Get the first review and pair it with the prompt
review = df['Review'].values[0]
prompt = base_prompt + review
print(prompt)

Summarize the sentiment and most important points in the following user review for a phone company named Megatelco.  You will evaluate the sentiment, theme (in 2 words or less), count the number of words, and suggest an action (in 2 words or less).Format the output in a table with columns: Sentiment, Theme, Word Count, Suggested Action. Review: I purchased a Megatelco phone last week, and it has sick performance. The camera quality is great, and the battery life is long. Overall, a solid 4-star experience.


### 🔧 Try It Yourself — Part 2

1. Create a new version of the prompt that adds two additional columns to the output.   (this columns should be numeric values you can visualize at a later time)

### In Your Response:
1. Why did you pick the two columns that you added?  What business insight would they provide?

In [28]:
base_prompt = (
    "Summarize the sentiment and most important points in the following user review for "
    "a phone company named Megatelco. "
    "Evaluate the sentiment, theme (2 words or less), word count, and suggested action (2 words or less). "
    "Also include two numeric columns: Sentiment Score (from -1 to 1) and Urgency Score (from 1 to 5). "
    "Format the output as a table with these columns: "
    "Sentiment, Theme, Word Count, Suggested Action, Sentiment Score, Urgency Score. "
    "Review: ")

review = df['Review'].iloc[0]
prompt = base_prompt + review

print(prompt)

Summarize the sentiment and most important points in the following user review for a phone company named Megatelco. Evaluate the sentiment, theme (2 words or less), word count, and suggested action (2 words or less). Also include two numeric columns: Sentiment Score (from -1 to 1) and Urgency Score (from 1 to 5). Format the output as a table with these columns: Sentiment, Theme, Word Count, Suggested Action, Sentiment Score, Urgency Score. Review: I purchased a Megatelco phone last week, and it has sick performance. The camera quality is great, and the battery life is long. Overall, a solid 4-star experience.


### ✍️ Your Response: 🔧
1. I added Sentiment Score and Urgency Score as numeric columns. Sentiment Score allows us to quantify customer feelings and visualize overall satisfaction trends. Urgency Score helps identify which issues need immediate attention. Together, these provide actionable business insight by showing not only how customers feel but also how critical their concerns are.

# Part 3: Connect with the API and Test

### What you are going to do:
- Create a connection to Gemini
- Run a test prompt
- Pass the full collection of reviews to the API (either in a batch or one-by-one in a loop)
- Format the output in a dataframe.   

### Do the following
- Go to https://aistudio.google.com/api-keys
- Click on the `Get API key` link on the bottom left corner
- Copy the value into the box below
- Send the first customer review to Gemini for analysis, then view the result

**Things to notice:**
- Is there any limit to the number of free requests you can make to Gemini?  (without payment)

In [29]:
# Configure the API key
API_KEY = '##Paste your API key here##'
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('models/gemini-2.5-flash')
print("Gemini API configured successfully.")

Gemini API configured successfully.


In [30]:
import google.generativeai as genai

API_KEY = "PASTE_YOUR_REAL_API_KEY_HERE"
genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-1.5-flash")
print("Gemini API configured successfully.")

Gemini API configured successfully.


### 🔧 Try It Yourself — Part 3
Ask Gemini to help you loop through the reviews, one by one, and format them into a dataframe. Use the following steps:
1. Build a full prompt by combining the base prompt that you created above with one or more reviews
2. Pass the full prompt to Gemini
3. Format the response into a dataframe
4. If you are processing one row at a time, pass the next prompt (in a loop) until you have processed all of the reviews
5. Show the final dataframe using `df.head()`

### In Your Response:
1. How does the output of the LLM compare to the output we saw in week 13 from VADER or TextBlob?

In [31]:
results = []

structured_prompt = """
Summarize the sentiment and most important points in the following user review for a phone company named Megatelco.

Return exactly one line in this format:
Sentiment,Theme,Word Count,Suggested Action,Sentiment Score,Urgency Score

Rules:
- Sentiment = Positive, Negative, or Neutral
- Theme = 2 words or less
- Word Count = integer
- Suggested Action = 2 words or less
- Sentiment Score = numeric from -1 to 1
- Urgency Score = integer from 1 to 5

Review:
"""

for review in df['Review'][:20]:   # use 20 first so it runs faster
    full_prompt = structured_prompt + str(review)

    try:
        response = model.generate_content(full_prompt)
        output = response.text.strip()

        parts = [x.strip() for x in output.split(",")]

        if len(parts) == 6:
            results.append({
                "Review": review,
                "Sentiment": parts[0],
                "Theme": parts[1],
                "Word Count": parts[2],
                "Suggested Action": parts[3],
                "Sentiment Score": parts[4],
                "Urgency Score": parts[5] })
        else:
            results.append({
                "Review": review,
                "Sentiment": "Parse Error",
                "Theme": "",
                "Word Count": "",
                "Suggested Action": "",
                "Sentiment Score": "",
                "Urgency Score": ""})

    except Exception as e:
        results.append({
            "Review": review,
            "Sentiment": f"ERROR: {e}",
            "Theme": "",
            "Word Count": "",
            "Suggested Action": "",
            "Sentiment Score": "",
            "Urgency Score": ""})

results_df = pd.DataFrame(results)
results_df.head()


,Review,Sentiment,Theme,Word Count,Suggested Action,Sentiment Score,Urgency Score
0,"I purchased a Megatelco phone last week, and i...",ERROR: 400 POST https://generativelanguage.goo...,,,,,
1,"My Megatelco phone arrived fast, but I've noti...",ERROR: 400 POST https://generativelanguage.goo...,,,,,
2,Megatelco offers top-notch phones! I'm happy w...,ERROR: 400 POST https://generativelanguage.goo...,,,,,
3,I've been using Megatelco phones for a while n...,ERROR: 400 POST https://generativelanguage.goo...,,,,,
4,"Regrettably, my experience with Megatelco was ...",ERROR: 400 POST https://generativelanguage.goo...,,,,,


### ✍️ Your Response: 🔧
1. The LLM output is more detailed and context-aware than VADER or TextBlob. VADER and TextBlob mainly measure positive or negative tone, while the LLM can also identify themes and suggest actions. This makes the LLM more useful for business decisions because it provides richer and more practical insights from the reviews



# Part 4: Visualize the Output
### What you are going to do:
- create visualizations to summarize the customer reviews.

## Why this matters:
If we have thousands of reviews, you will need to summarize them for management use.  Each chart should tell a distinct story about the customer feedback, themes and suggested action items.  

### 🔧 Try It Yourself — Part 4
Create at least four visualizations to answer the following questions:
1. What are the main themes?
2. For each theme, what is the sentiment associated with the theme?
3. What are the action items that should be taken to reduce churn?
4. Add one or more visualizations that will show the insights from the fields that you added in part 2.  

### In Your Response:
1. Why did you pick the charts or image types for each of the four visualizations?  

In [32]:
import matplotlib.pyplot as plt
import pandas as pd

# -----------------------------
# Clean the dataframe
# -----------------------------
clean_df = results_df.copy()

clean_df = clean_df[~clean_df["Sentiment"].astype(str).str.startswith("ERROR", na=False)]
clean_df = clean_df[clean_df["Sentiment"] != "Parse Error"]

# Clean text columns
clean_df["Theme"] = clean_df["Theme"].astype(str).str.strip()
clean_df["Sentiment"] = clean_df["Sentiment"].astype(str).str.strip()
clean_df["Suggested Action"] = clean_df["Suggested Action"].astype(str).str.strip()

# Remove blank values
clean_df = clean_df[clean_df["Theme"] != ""]
clean_df = clean_df[clean_df["Sentiment"] != ""]
clean_df = clean_df[clean_df["Suggested Action"] != ""]

# Convert numeric columns
clean_df["Word Count"] = pd.to_numeric(clean_df["Word Count"], errors="coerce")
clean_df["Sentiment Score"] = pd.to_numeric(clean_df["Sentiment Score"], errors="coerce")
clean_df["Urgency Score"] = pd.to_numeric(clean_df["Urgency Score"], errors="coerce")

# -----------------------------
# Chart 1: Main themes
# -----------------------------
theme_counts = clean_df["Theme"].value_counts().head(10)

if len(theme_counts) > 0:
    plt.figure(figsize=(10,6))
    plt.bar(theme_counts.index, theme_counts.values)
    plt.title("Top Themes in Customer Reviews")
    plt.xlabel("Theme")
    plt.ylabel("Number of Reviews")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No theme data available for Chart 1")

# -----------------------------
# Chart 2: Sentiment by theme
# -----------------------------
theme_sentiment = (
    clean_df.groupby(["Theme", "Sentiment"])
    .size()
    .unstack(fill_value=0)
    .head(10)
)

if len(theme_sentiment) > 0:
    theme_sentiment.plot(kind="bar", figsize=(10,6))
    plt.title("Sentiment by Theme")
    plt.xlabel("Theme")
    plt.ylabel("Number of Reviews")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No theme/sentiment data available for Chart 2")

# -----------------------------
# Chart 3: Suggested actions
# -----------------------------
action_counts = clean_df["Suggested Action"].value_counts().head(10)

if len(action_counts) > 0:
    plt.figure(figsize=(10,6))
    plt.bar(action_counts.index, action_counts.values)
    plt.title("Most Common Suggested Actions")
    plt.xlabel("Suggested Action")
    plt.ylabel("Number of Reviews")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No action data available for Chart 3")

# -----------------------------
# Chart 4: Sentiment Score vs Urgency Score
# -----------------------------
scatter_df = clean_df.dropna(subset=["Sentiment Score", "Urgency Score"])

if len(scatter_df) > 0:
    plt.figure(figsize=(10,6))
    plt.scatter(scatter_df["Sentiment Score"], scatter_df["Urgency Score"])
    plt.title("Sentiment Score vs Urgency Score")
    plt.xlabel("Sentiment Score")
    plt.ylabel("Urgency Score")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric data available for Chart 4")

clean_df.head()

No theme data available for Chart 1
No theme/sentiment data available for Chart 2
No action data available for Chart 3
No numeric data available for Chart 4


,Review,Sentiment,Theme,Word Count,Suggested Action,Sentiment Score,Urgency Score


### ✍️ Your Response: 🔧
1. I used a bar chart for the main themes because it clearly shows which topics appear most often in the reviews.
2. I used a grouped bar chart for sentiment by theme because it makes it easy to compare whether each theme is mostly positive, negative, or neutral.
3. I used a bar chart for suggested actions because it shows which actions the company should prioritize to reduce churn.
4. I used a scatter plot for Sentiment Score and Urgency Score because it helps show the relationship between customer emotion and how serious the issue is.


## 🔧 Part 5: Reflection (100 words or less)

In this lab you connected to an LLM API to request summarization of customer reviews.  

Use the cell below to answer the following questions:

1. What was the elapsed time to collect the LLM responses to all of the requests?  How long would it take to process 1,000 requests?
2. What are the advantages and disadvantes of using Gemini versus VADER or TextBlob, which we used in Lab 13?  
3. Write a prompt that you could use to an LLM to create a business strategy and business plan to improve customer churn.   

### ✍️ Your Response: 🔧
1. The elapsed time depends on the number of reviews and API speed. If each request takes about 1 to 2 seconds, then 1,000 requests would take about 17 to 33 minutes.
2. Gemini is better than VADER or TextBlob for understanding context, themes, and suggested actions, but it is slower and requires API access. VADER and TextBlob are faster, simpler, and cheaper.
3. A useful prompt would be: Analyze these customer reviews, identify the main causes of churn, and create a business strategy with action steps to improve retention and customer satisfaction.


# Export Your Notebook to Submit in Canvas
Use the instructions from Lab 1

In [33]:
!jupyter nbconvert --to html "lab_14_api.ipynb"

[NbConvertApp] WARNING | pattern 'lab_14_api.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    